# University AI Assistant Fine-Tuning using LoRA, QLoRA & DPO

## Project Overview

This project demonstrates how to fine-tune a Large Language Model (LLM) for a
University AI Assistant using modern Parameter-Efficient Fine-Tuning (PEFT)
techniques.

The notebook implements and compares three state-of-the-art fine-tuning
approaches:

- Supervised Fine-Tuning (SFT) using LoRA
- Supervised Fine-Tuning (SFT) using QLoRA
- Direct Preference Optimization (DPO)

The final models Adapter and Merged are published to Hugging Face Hub.

Training and Test data for Models is Synthetic data prepared with the help of Gemini Flash(Extended Thinking) and Chatgpt


## Project Workflow

```text
Raw Excel Dataset
        │
        ▼
Dataset Cleaning
        │
        ▼
Chat Message Creation
        │
        ▼
Chat Template Formatting
        │
        ▼
SFT Dataset
        │
        ├───────────────┐
        ▼               ▼
     LoRA            QLoRA
        │               │
        └───────┬───────┘
                ▼
        DPO Fine-Tuning
                │
                ▼
         Model Evaluation
                │
                ▼
       Hugging Face Hub

## Enviornment Setup

In [ ]:
# ==============================================================================
# Install Required Libraries
# ==============================================================================
#
# The kernel is restarted automatically so the newly installed packages
# are loaded correctly.
#
# After the restart, continue execution from Cell 3.
# ==============================================================================

%pip install -q -U \
    unsloth \
    trl \
    openpyxl \
    mergekit

import os
os._exit(0)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 MB 24.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.2/102.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.7/354.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

In [1]:
# ==============================================================================
# Verify Python Environment to avoid packages not getting installed
# ==============================================================================

import importlib
packages = [
    "torch",
    "transformers",
    "peft",
    "trl",
    "unsloth",
]
print("=" * 80)
print("Environment Verification")
print("=" * 80)
for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "Unknown")
        print(f"{package:<15}: {version}")
    except Exception as e:
        print(f"{package:<15}: NOT AVAILABLE")
        print(e)
print("=" * 80)

Environment Verification
torch          : 2.10.0+cu128
transformers   : 5.5.0


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


peft           : 0.19.1
trl            : 0.24.0
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be diffe

🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth        : 2026.7.2


In [2]:
# ------------------------------------------------------------------------------
# Standard Python Libraries
# ------------------------------------------------------------------------------
import os
import random
import warnings
import platform
from pathlib import Path

# Data Processing Libraries
import numpy as np
import pandas as pd

# Deep Learning
import torch

# Unsloth
# IMPORTANT: Always import Unsloth BEFORE Transformers, PEFT and TRL.
import unsloth
from unsloth import FastLanguageModel

# Hugging Face Libraries
import datasets
from datasets import Dataset, load_from_disk
from huggingface_hub import login

# Transformers
import transformers

# Suppress unnecessary warnings
warnings.filterwarnings("ignore")

In [3]:
# ------------------------------------------------------------------------------
# Project Configuration
# ------------------------------------------------------------------------------

print("=" * 80)
print("Project Configuration")
print("=" * 80)

# Project Information
PROJECT_NAME = "University AI Assistant"
PROJECT_VERSION = "2026"

# Base Model
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct"
# Other supported examples:
#
# MODEL_NAME = "unsloth/Qwen3-4B-Base"
# MODEL_NAME = "unsloth/gemma-3-4b-it"

# Hugging Face Configuration
HF_USER_NAME = "Akay2026"

# Dataset Configuration

# System prompt used while creating the SFT dataset.
SYSTEM_PROMPT = (
    "You are Global Tech University's AI Student Assistant."
)
# Maximum sequence length used while formatting the dataset.
# The same value will later be used during model training.
MAX_SEQ_LENGTH = 1024

# ------------------------------------------------------------------------------
# Folder Structure
# ------------------------------------------------------------------------------
# Folder containing the input Excel datasets.
# DATA_DIR = Path( "/kaggle/input/datasets/akay765/university-ai-assistant-dataset")
DATA_DIR = Path(
    "/kaggle/input/datasets/akay765/university-ai-assistant-dataset-v2"
)

# Folder used to store processed Hugging Face datasets.
OUTPUT_DIR = Path("./output")

# Folder used to store trained models.
MODEL_DIR = Path("./models")

# Folder used to store logs.
LOG_DIR = Path("./logs")

# Create folders if they do not already exist.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Random Seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Display Configuration
print(f"Project Name        : {PROJECT_NAME}")
print(f"Project Version     : {PROJECT_VERSION}")
print(f"Base Model          : {MODEL_NAME}")
print(f"Hugging Face User   : {HF_USER_NAME}")
print(f"Maximum Seq Length  : {MAX_SEQ_LENGTH}")
print(f"Dataset Folder      : {DATA_DIR}")
print(f"Output Folder       : {OUTPUT_DIR}")
print(f"Model Folder        : {MODEL_DIR}")
print(f"Random Seed         : {SEED}")
print("=" * 80)

Project Configuration
Project Name        : University AI Assistant
Project Version     : 2026
Base Model          : unsloth/Llama-3.2-1B-Instruct
Hugging Face User   : Akay2026
Maximum Seq Length  : 1024
Dataset Folder      : /kaggle/input/datasets/akay765/university-ai-assistant-dataset-v2
Output Folder       : output
Model Folder        : models
Random Seed         : 42


In [4]:
# ==============================================================================
# Hugging Face Authentication
# ==============================================================================
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

print("Hugging Face Authentication")

user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_Write")

from huggingface_hub import login

login(secret_value_0)    # Paste your HF  Token

Hugging Face Authentication


## Load Base Model

In [5]:
# ==============================================================================
# Load Base Model and Tokenizer
# ==============================================================================
#
# This section loads the pre-trained base model and its tokenizer.
#
# The loaded tokenizer will be reused throughout the notebook for:
#
#   • Dataset Preparation
#   • LoRA Training
#   • QLoRA Training
#   • DPO Training
#   • Inference
#
# The base model loaded here is the original pre-trained model.
# LoRA adapters will be attached later during training.
#
# ==============================================================================

print("=" * 80)
print("Loading Base Model")
print("=" * 80)

# ------------------------------------------------------------------------------
# Load Model and Tokenizer
# ------------------------------------------------------------------------------

model, tokenizer = FastLanguageModel.from_pretrained(

    # Base model
    model_name=MODEL_NAME,

    # Maximum context length
    max_seq_length=MAX_SEQ_LENGTH,

    # Notebook starts with full precision model.
    # QLoRA will later reload the same model in 4-bit.
    load_in_4bit=False,

)

print("\n✓ Base model loaded successfully.")
print(f"\nModel Name : {MODEL_NAME}")
print(f"Maximum Sequence Length : {MAX_SEQ_LENGTH}")
print("=" * 80)

Loading Base Model
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.



✓ Base model loaded successfully.

Model Name : unsloth/Llama-3.2-1B-Instruct
Maximum Sequence Length : 1024


## Dataset Preparation

In [6]:
# ==============================================================================
# Load Input Datasets
# ==============================================================================
#
# This section loads the Excel datasets required for:
#
#   1. Supervised Fine-Tuning (SFT)
#   2. Direct Preference Optimization (DPO)
#   3. Model Testing
#
# ==============================================================================

print("=" * 80)
print("Loading Input Datasets")
print("=" * 80)

# Verify Dataset Files
required_files = [
    # "02_SFT_Dataset.xlsx",
    # "03_DPO_Dataset.xlsx",
    # "04_Test_Dataset.xlsx",

    "University_SFT_Dataset.xlsx",
    "University_DPO_Dataset.xlsx",
    "University_TEST_Dataset.xlsx",
]

missing_files = []

for file_name in required_files:
    file_path = DATA_DIR / file_name
    if file_path.exists():
        print(f"✓ {file_name}")
    else:
        print(f"✗ {file_name}")
        missing_files.append(file_name)

# Stop execution if any file is missing

if len(missing_files) > 0:
    raise FileNotFoundError(
        f"\nMissing dataset files:\n{missing_files}"
    )

print()

# Load Excel Files

sft_df = pd.read_excel(
    # DATA_DIR / "02_SFT_Dataset.xlsx"
    DATA_DIR /"University_SFT_Dataset.xlsx"
)

dpo_df = pd.read_excel(
    # DATA_DIR / "03_DPO_Dataset.xlsx"
    DATA_DIR /"University_DPO_Dataset.xlsx"
)

test_df = pd.read_excel(
    # DATA_DIR / "04_Test_Dataset.xlsx"
    DATA_DIR /"University_TEST_Dataset.xlsx"
)

print("Datasets loaded successfully.\n")

# ------------------------------------------------------------------------------
# Display Dataset Summary
# ------------------------------------------------------------------------------

summary = pd.DataFrame({
    "Dataset": [ "SFT","DPO","Test",],
    "Rows": [len(sft_df),len(dpo_df),len(test_df),],
    "Columns": [len(sft_df.columns),len(dpo_df.columns),len(test_df.columns),]

})

display(summary)
print("=" * 80)

Loading Input Datasets
✓ University_SFT_Dataset.xlsx
✓ University_DPO_Dataset.xlsx
✓ University_TEST_Dataset.xlsx

Datasets loaded successfully.



,Dataset,Rows,Columns
0,SFT,1000,4
1,DPO,1000,5
2,Test,500,4


In [7]:
# ==============================================================================
# Validate Input Datasets
# ==============================================================================
#
# This section performs basic validation on all datasets before preprocessing.
#
# The following checks are performed:
#
#   • Dataset dimensions
#   • Column names
#   • Data types
#   • Missing values
#   • Duplicate records
#   • Sample records
#
# ==============================================================================

print("=" * 80)
print("Dataset Validation")
print("=" * 80)

datasets = {
    "SFT Dataset": sft_df,
    "DPO Dataset": dpo_df,
    "Test Dataset": test_df,
}

for dataset_name, df in datasets.items():

    print("\n" + "=" * 80)
    print(dataset_name)
    print("=" * 80)

    # Dataset Shape
    print(f"Rows    : {df.shape[0]}")
    print(f"Columns : {df.shape[1]}")

    # Column Names
    print("\nColumns")
    for column in df.columns:
        print(f" • {column}")

    # Data Types
    print("\nData Types")
    display(df.dtypes.to_frame(name="Data Type"))

    # Missing Values
    print("\nMissing Values")
    missing_values = df.isnull().sum()
    display(missing_values.to_frame(name="Missing Count"))

    # Duplicate Rows
    duplicate_rows = df.duplicated().sum()
    print(f"\nDuplicate Rows : {duplicate_rows}")

    # Preview Dataset
    print("\nFirst 3 Records")
    display(df.head(3))

print("=" * 80)
print("Dataset validation completed.")
print("=" * 80)

Dataset Validation

SFT Dataset
Rows    : 1000
Columns : 4

Columns
 • id
 • category
 • prompt
 • response

Data Types


,Data Type
id,object
category,object
prompt,object
response,object



Missing Values


,Missing Count
id,0
category,0
prompt,0
response,0



Duplicate Rows : 0

First 3 Records


...


Missing Values


,Missing Count
id,0
category,0
question,0
expected_answer,0



Duplicate Rows : 0

First 3 Records


,id,category,question,expected_answer
0,TEST-0001,Admissions,Could someone explain what happens if a studen...,**Official Advisor Guidance:** When addressing...
1,TEST-0002,Transfer Admissions,What are the specific requirements and documen...,**Official Advisor Guidance:** When addressing...
2,TEST-0003,International Admissions,Is there an established protocol for appealing...,**Official Advisor Guidance:** When addressing...


Dataset validation completed.


In [8]:
# ==============================================================================
# Clean Input Datasets
# ==============================================================================
#
# This section performs basic data cleaning before preparing the datasets for
# fine-tuning.
#
# Cleaning steps:
#
#   • Remove duplicate records
#   • Remove rows with missing values
#   • Reset DataFrame index
#   • Display cleaning summary
#
# ==============================================================================

print("=" * 80)
print("Cleaning Input Datasets")
print("=" * 80)

datasets = {
    "SFT Dataset": sft_df,
    "DPO Dataset": dpo_df,
    "Test Dataset": test_df,
}

cleaned_datasets = {}

summary = []

for dataset_name, df in datasets.items():

    original_rows = len(df)

    # Remove Duplicate Records
    duplicate_rows = df.duplicated().sum()
    df = df.drop_duplicates()

    # Remove Missing Values
    missing_rows = df.isnull().any(axis=1).sum()
    df = df.dropna()

    # Remove leading/trailing whitespace from all text columns
    text_columns = df.select_dtypes(include="object").columns

    for column in text_columns:
        df[column] = df[column].str.strip()

    # Reset Index
    df = df.reset_index(drop=True)
    final_rows = len(df)

    cleaned_datasets[dataset_name] = df

    summary.append({
        "Dataset": dataset_name,
        "Original Rows": original_rows,
        "Duplicates Removed": duplicate_rows,
        "Rows with Missing Values": missing_rows,
        "Final Rows": final_rows
    })

# Update DataFrames
sft_df = cleaned_datasets["SFT Dataset"]
dpo_df = cleaned_datasets["DPO Dataset"]
test_df = cleaned_datasets["Test Dataset"]

# Cleaning Summary
summary_df = pd.DataFrame(summary)

display(summary_df)

print("=" * 80)
print("Dataset cleaning completed successfully.")
print("=" * 80)

Cleaning Input Datasets


,Dataset,Original Rows,Duplicates Removed,Rows with Missing Values,Final Rows
0,SFT Dataset,1000,0,0,1000
1,DPO Dataset,1000,0,0,1000
2,Test Dataset,500,0,0,500


Dataset cleaning completed successfully.


In [9]:
# ==============================================================================
# Prepare Training Datasets for LLM fine-tuning.
# ==============================================================================
#
# SFT Dataset
# -----------
# Creates:
#   • messages : Chat conversation
#   • text     : Model input using the tokenizer's chat template
#
# DPO Dataset
# -----------
# Creates:
#   • prompt
#   • chosen
#   • rejected
#
# ==============================================================================

print("=" * 80)
print("Preparing Training Datasets")
print("=" * 80)

# Build Chat Conversation

def build_messages(row):

    return [ 
        { "role": "system","content": SYSTEM_PROMPT,},
        { "role": "user", "content": row["prompt"],},
        { "role": "assistant","content": row["response"], },
    ]


# Create chat messages
sft_df["messages"] = sft_df.apply(
    build_messages,
    axis=1,
)

# Convert Chat Messages to Model Text
sft_df["text"] = sft_df["messages"].apply(

    lambda messages: tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
)

# Prepare DPO Dataset
dpo_processed = pd.DataFrame({

    "prompt": dpo_df["prompt"],
    "chosen": dpo_df["chosen_response"],
    "rejected": dpo_df["rejected_response"],

})

print("✓ SFT dataset prepared.")
print("✓ DPO dataset prepared.")
print("=" * 80)

Preparing Training Datasets
✓ SFT dataset prepared.
✓ DPO dataset prepared.


In [10]:
# ==============================================================================
# Create Hugging Face Datasets
# ==============================================================================
#
# Convert the processed pandas DataFrames into Hugging Face Dataset objects.
#
# Only the columns required for training are retained.
#
# SFT Dataset
#   • text
#
# DPO Dataset
#   • prompt
#   • chosen
#   • rejected
#
# Test Dataset
#   • Used later for inference and evaluation
#
# ==============================================================================

print("=" * 80)
print("Creating Hugging Face Datasets")
print("=" * 80)

# ------------------------------------------------------------------------------
# Keep only the required columns
# ------------------------------------------------------------------------------

sft_processed = sft_df[["text"]]

dpo_processed = dpo_processed[
    ["prompt", "chosen", "rejected"]
]

test_processed = test_df.copy()

# ------------------------------------------------------------------------------
# Convert Pandas DataFrames to Hugging Face Datasets
# ------------------------------------------------------------------------------

sft_dataset = Dataset.from_pandas(
    sft_processed,
    preserve_index=False,
)

dpo_dataset = Dataset.from_pandas(
    dpo_processed,
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test_processed,
    preserve_index=False,
)

# ------------------------------------------------------------------------------
# Display Dataset Summary
# ------------------------------------------------------------------------------

summary = pd.DataFrame({

    "Dataset": ["SFT","DPO","Test",],

    "Records": [len(sft_dataset),len(dpo_dataset),len(test_dataset),],

    "Columns": [
        list(sft_dataset.features.keys()),
        list(dpo_dataset.features.keys()),
        list(test_dataset.features.keys()),
    ],

})

display(summary)

print()
print("✓ Hugging Face datasets created successfully.")
print("=" * 80)

Creating Hugging Face Datasets


,Dataset,Records,Columns
0,SFT,1000,[text]
1,DPO,1000,"[prompt, chosen, rejected]"
2,Test,500,"[id, category, question, expected_answer]"



✓ Hugging Face datasets created successfully.


In [11]:
# ==============================================================================
# Save and Verify Hugging Face Datasets
# ==============================================================================
#
# Save the processed datasets to disk so that they can be reused without
# repeating the preprocessing steps.
#
# After saving, reload each dataset to verify that it was written correctly.
#
# ==============================================================================

print("=" * 80)
print("Saving Hugging Face Datasets")
print("=" * 80)

# ------------------------------------------------------------------------------
# Define Output Locations
# ------------------------------------------------------------------------------

SFT_DATASET_DIR = OUTPUT_DIR / "sft_dataset"
DPO_DATASET_DIR = OUTPUT_DIR / "dpo_dataset"
TEST_DATASET_DIR = OUTPUT_DIR / "test_dataset"

# ------------------------------------------------------------------------------
# Save Datasets
# ------------------------------------------------------------------------------

sft_dataset.save_to_disk(SFT_DATASET_DIR)
dpo_dataset.save_to_disk(DPO_DATASET_DIR)
test_dataset.save_to_disk(TEST_DATASET_DIR)

print("✓ SFT Dataset saved.")
print("✓ DPO Dataset saved.")
print("✓ Test Dataset saved.")
print()

# ------------------------------------------------------------------------------
# Reload Saved Datasets
# ------------------------------------------------------------------------------

sft_dataset = load_from_disk(SFT_DATASET_DIR)
dpo_dataset = load_from_disk(DPO_DATASET_DIR)
test_dataset = load_from_disk(TEST_DATASET_DIR)
print("✓ Dataset verification completed.")

print()

# ------------------------------------------------------------------------------
# Display Dataset Summary
# ------------------------------------------------------------------------------

summary = pd.DataFrame({

    "Dataset": [

        "SFT",

        "DPO",

        "Test",

    ],

    "Records": [

        len(sft_dataset),

        len(dpo_dataset),

        len(test_dataset),

    ],

    "Features": [

        list(sft_dataset.features.keys()),

        list(dpo_dataset.features.keys()),

        list(test_dataset.features.keys()),

    ],

})

display(summary)

print()

# ------------------------------------------------------------------------------
# Display Sample Records
# ------------------------------------------------------------------------------

print("=" * 80)
print("Sample SFT Record")
print("=" * 80)

display(sft_dataset[0])

print("=" * 80)
print("Sample DPO Record")
print("=" * 80)

display(dpo_dataset[0])

print("=" * 80)
print("Sample Test Record")
print("=" * 80)

display(test_dataset[0])

print("=" * 80)
print("Dataset preparation completed successfully.")
print("=" * 80)

Saving Hugging Face Datasets


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

✓ SFT Dataset saved.
✓ DPO Dataset saved.
✓ Test Dataset saved.

✓ Dataset verification completed.



...

Sample DPO Record


{'prompt': 'What is the official institutional framework, policy workflow, and compliance standard for managing admissions ensuring that all procedural risks are minimized and that the student maintains full structural alignment with university regulations. [Case Identifier Ref: #10001]',
 'chosen': '**Brief Overview:**\nNavigating the administrative matrix for Admissions requires a thorough understanding of localized department workflows and centralized registry rules. The institution maintains an active verification process to ensure that all student records, transactions, and milestones are logged with full audit trails, minimizing procedural delays and protecting student progression.\n\n**Detailed Advisor Explanation:**\nApplying structural regulation mapping code ADM-0001 directly to this administrative pipeline, Managing documentation within this specific context involves an intricate series of policy filters designed to validate the legitimacy, authenticity, and completeness of 

Sample Test Record


{'id': 'TEST-0001',
 'category': 'Admissions',
 'question': 'Could someone explain what happens if a student misses the window for conditional acceptance offer reviews due to unforeseen conflicts?',
 'expected_answer': "**Official Advisor Guidance:** When addressing conditional acceptance offer reviews inside the admissions framework, the university manual specifies that every record must maintain structural validation. The university approaches these scenarios through an objective administrative lens designed to enforce institutional bylaws while accommodating legitimate, third-party verified disruptions. When a student encounters a barrier in this operational branch, the registrar opens an inquiry file to audit the account history against system transaction logs. If a variance is justified under standard operating policies, a temporary authorization is granted, allowing the student to continue their milestones without incurring late fees or academic holds. As a practical advisory rec

Dataset preparation completed successfully.


## LoRA Training

In [12]:
# ==============================================================================
# Training Configuration
# ==============================================================================
#
# This section defines all hyperparameters used during Supervised Fine-Tuning
# (SFT).
#
# The same configuration will be used for both:
#
#   • LoRA
#   • QLoRA
#
# This enables a fair comparison between the two approaches.
#
# ==============================================================================

from trl import SFTConfig

print("=" * 80)
print("Training Configuration")
print("=" * 80)

# ------------------------------------------------------------------------------
# LoRA Hyperparameters
# ------------------------------------------------------------------------------

LORA_R = 16

LORA_ALPHA = 16

LORA_DROPOUT = 0.0

TARGET_MODULES = [

    "q_proj",

    "k_proj",

    "v_proj",

    "o_proj",

    "gate_proj",

    "up_proj",

    "down_proj",

]

# ------------------------------------------------------------------------------
# SFT Training Hyperparameters
# ------------------------------------------------------------------------------

NUM_EPOCHS = 1

BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 2e-4

LOGGING_STEPS = 10

# ------------------------------------------------------------------------------
# Training Configuration
# ------------------------------------------------------------------------------

training_args = SFTConfig(

    # Directory for checkpoints and logs
    output_dir="./outputs_sft",

    # Maximum sequence length
    max_length=MAX_SEQ_LENGTH,

    # Training
    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,

    # Logging
    logging_steps=LOGGING_STEPS,

    # Save only the final checkpoint
    save_strategy="epoch",

    # Disable evaluation for this tutorial
    eval_strategy="no",

    # Use BF16 if supported; otherwise FP16
    bf16=torch.cuda.is_bf16_supported(),

    fp16=not torch.cuda.is_bf16_supported(),

    # Report only to notebook
    report_to="none",

    # Reproducibility
    seed=SEED,

)

print("✓ Training configuration created successfully.\n")

print(f"Epochs                    : {NUM_EPOCHS}")
print(f"Batch Size                : {BATCH_SIZE}")
print(f"Learning Rate             : {LEARNING_RATE}")
print(f"Gradient Accumulation     : {GRADIENT_ACCUMULATION_STEPS}")
print(f"LoRA Rank (r)             : {LORA_R}")
print(f"LoRA Alpha                : {LORA_ALPHA}")
print(f"LoRA Dropout              : {LORA_DROPOUT}")

print("=" * 80)

Training Configuration
✓ Training configuration created successfully.

Epochs                    : 1
Batch Size                : 2
Learning Rate             : 0.0002
Gradient Accumulation     : 4
LoRA Rank (r)             : 16
LoRA Alpha                : 16
LoRA Dropout              : 0.0


### Verify Training Dataset

In [13]:
# ==============================================================================
# Verify Training Dataset
# ==============================================================================
#
# Verify that the processed Hugging Face datasets have the expected structure
# before starting model training.
#
# This step helps detect dataset issues early and avoids failures during
# SFT/DPO training.
#
# ==============================================================================

print("=" * 80)
print("Verifying Training Datasets")
print("=" * 80)

# ------------------------------------------------------------------------------
# SFT Dataset
# ------------------------------------------------------------------------------

print("\nSFT Dataset")

print("-" * 80)

print(f"Records : {len(sft_dataset)}")

print(f"Columns : {list(sft_dataset.features.keys())}")

print("\nFirst Training Sample\n")

print(sft_dataset[0]["text"])

# ------------------------------------------------------------------------------
# DPO Dataset
# ------------------------------------------------------------------------------

print("\n" + "=" * 80)

print("DPO Dataset")

print("-" * 80)

print(f"Records : {len(dpo_dataset)}")

print(f"Columns : {list(dpo_dataset.features.keys())}")

print("\nFirst Preference Sample\n")

print("Prompt")
print("-" * 80)
print(dpo_dataset[0]["prompt"])

print("\nChosen")
print("-" * 80)
print(dpo_dataset[0]["chosen"])

print("\nRejected")
print("-" * 80)
print(dpo_dataset[0]["rejected"])

print("=" * 80)

print("Dataset verification completed successfully.")

print("=" * 80)

Verifying Training Datasets

SFT Dataset
--------------------------------------------------------------------------------
Records : 1000
Columns : ['text']

First Training Sample

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 14 Jul 2026

You are Global Tech University's AI Student Assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the official institutional framework, policy workflow, and compliance standard for managing admissions ensuring that all procedural risks are minimized and that the student maintains full structural alignment with university regulations. [Case Identifier Ref: #10001]<|eot_id|><|start_header_id|>assistant<|end_header_id|>

**Brief Overview:**
Navigating the administrative matrix for Admissions requires a thorough understanding of localized department workflows and centralized registry rules. The institution maintains an active verification process to ensure that all student re

In [14]:
# ==============================================================================
# Common SFT Training Function (LoRA / QLoRA)
# ==============================================================================
#
# This function performs Supervised Fine-Tuning (SFT) using either:
#
#   • LoRA   (load_in_4bit=False)
#   • QLoRA  (load_in_4bit=True)
#
# Responsibilities
# ----------------
# ✓ Load the base model
# ✓ Attach LoRA adapters
# ✓ Fine-tune the model
# ✓ Save the trained adapter locally
#
# This function intentionally DOES NOT:
#
# ✗ Upload to Hugging Face
# ✗ Merge adapters
# ✗ Perform inference
#
# Those operations are performed in separate notebook cells.
#
# ==============================================================================

from trl import SFTTrainer


def train_sft(
    use_4bit: bool,
    local_model_name: str,
):

    print("=" * 80)
    print(
        f"Training {'QLoRA' if use_4bit else 'LoRA'} Model"
    )
    print("=" * 80)

    # ------------------------------------------------------------------
    # Load Base Model
    # ------------------------------------------------------------------

    model, tokenizer = FastLanguageModel.from_pretrained(

        model_name=MODEL_NAME,

        max_seq_length=MAX_SEQ_LENGTH,

        load_in_4bit=use_4bit,

    )

    # ------------------------------------------------------------------
    # Attach LoRA Adapters
    # ------------------------------------------------------------------

    model = FastLanguageModel.get_peft_model(

        model,

        r=LORA_R,

        lora_alpha=LORA_ALPHA,

        lora_dropout=LORA_DROPOUT,

        target_modules=TARGET_MODULES,

    )

    # ------------------------------------------------------------------
    # Create Trainer
    # ------------------------------------------------------------------

    trainer = SFTTrainer(

        model=model,

        processing_class=tokenizer,

        train_dataset=sft_dataset,

        args=training_args,

    )

    # ------------------------------------------------------------------
    # Train
    # ------------------------------------------------------------------

    trainer.train()

    # ------------------------------------------------------------------
    # Save Adapter
    # ------------------------------------------------------------------

    model_path = MODEL_DIR / local_model_name

    trainer.model.save_pretrained(model_path)

    tokenizer.save_pretrained(model_path)

    print()

    print(f"✓ Adapter saved to")

    print(model_path)

    print()

    return {
        "trainer": trainer,
        "model": trainer.model,
        "tokenizer": tokenizer,
        "path": model_path,
    
        # Metadata
        "model_name": local_model_name,
        "base_model": MODEL_NAME,
        "model_type": "QLoRA" if use_4bit else "LoRA",
        "repo_name": local_model_name,
    }

### Train LoRA Model

In [15]:
# ==============================================================================
# Train LoRA Model
# ==============================================================================
#
# This section fine-tunes the base model using LoRA adapters.
#
# LoRA trains only a small number of additional parameters while keeping the
# original model weights frozen.
#
# The trained adapter is saved locally for later evaluation and inference.
#
# ==============================================================================

print("=" * 80)
print("Training LoRA Model")
print("=" * 80)

# ------------------------------------------------------------------------------
# Train LoRA Model
# ------------------------------------------------------------------------------

lora = train_sft(
    use_4bit=False,
    local_model_name="University-AI-LoRA",
)

print()

print("✓ LoRA training completed successfully.")

print()

print(f"Model saved at:\n{lora['path']}")
print("=" * 80)

Training LoRA Model
Training LoRA Model
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.
Unsloth 2026.7.2 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.685995
20,2.795690
30,1.670321
40,0.630091
50,0.186190
60,0.096109
70,0.077392
80,0.066674
90,0.062806
100,0.060936


Unsloth: Restored added_tokens_decoder metadata in ./outputs_sft/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in models/University-AI-LoRA/tokenizer_config.json.



✓ Adapter saved to
models/University-AI-LoRA


✓ LoRA training completed successfully.

Model saved at:
models/University-AI-LoRA


### Common Inference Function

### Enhanced Common Inference function

This function which works for single and multiple questions 

In [16]:
def generate_response(
    model_path,
    questions,
    num_questions_to_run=None,
    max_new_tokens=200,
    temperature=0.7,
):
    """
    Generate responses from a saved fine-tuned model.

    Parameters
    ----------
    model_path : str
        Path to the saved LoRA, QLoRA, or DPO model.

    questions : str | list[str]
        A single question or a list of questions.

    num_questions_to_run : int, optional
        Number of questions to process. If None, all questions
        are processed. If the value exceeds the number of available
        questions, all questions are processed.

    max_new_tokens : int, default=200
        Maximum number of new tokens to generate.

    temperature : float, default=0.7
        Sampling temperature used during text generation.

    Returns
    -------
    str | list[str]
        Returns a single response when a single question is provided.
        Returns a list of responses when multiple questions are provided.
    """

    # --------------------------------------------------------------------------
    # Validate & Normalize Input
    # --------------------------------------------------------------------------

    single_question = False

    if isinstance(questions, str):
        questions = [questions]
        single_question = True

    elif not isinstance(questions, list):
        raise TypeError(
            "questions must be either a string or a list of strings."
        )

    if num_questions_to_run is None:
        num_questions_to_run = len(questions)

    num_questions_to_run = min(
        num_questions_to_run,
        len(questions),
    )

    # --------------------------------------------------------------------------
    # Load Model
    # --------------------------------------------------------------------------

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(model_path),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )

    FastLanguageModel.for_inference(model)

    responses = []

    # --------------------------------------------------------------------------
    # Generate Responses
    # --------------------------------------------------------------------------

    for question in questions[:num_questions_to_run]:

        # ----------------------------------------------------------------------
        # Create Chat Prompt
        # ----------------------------------------------------------------------

        messages = [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": question,
            },
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        # ----------------------------------------------------------------------
        # Tokenize
        # ----------------------------------------------------------------------

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(model.device)

        # ----------------------------------------------------------------------
        # Generate Response
        # ----------------------------------------------------------------------

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

        response = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True,
        )

        # ----------------------------------------------------------------------
        # Remove Prompt from Output
        # ----------------------------------------------------------------------

        response = response.replace(
            prompt,
            "",
        ).strip()

        # Decode only the newly generated tokens
        # generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        
        # response = tokenizer.decode(
        #     generated_tokens,
        #     skip_special_tokens=True,
        # ).strip()
        
        # Remove any leading assistant tag if present
        # response = response.removeprefix("assistant").strip()
        
        responses.append(response)

    # --------------------------------------------------------------------------
    # Return Response(s)
    # --------------------------------------------------------------------------

    if single_question:
        return responses[0]

    return responses

### Test Lora with common Inference function

In [17]:
question = "How can I apply for a merit scholarship?"

response = generate_response(
    model_path=lora["path"],
    questions=question
)
print(question)
print(response)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-LoRA as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How can I apply for a merit scholarship?
system

Cutting Knowledge Date: December 2023
Today Date: 14 Jul 2026

You are Global Tech University's AI Student Assistant.user

How can I apply for a merit scholarship?assistant

Applying for a merit scholarship at our university involves an administrative approval process designed to protect institutional governance and balance academic accountability with operational efficiency. For the most accurate operational timing and course circuit compliance, we recommend that you consult the central regulatory board calendar and speak with a regional administrative officer directly.


### Common Hugging Face Adapter Upload Function

In [18]:
from huggingface_hub import create_repo

def upload_to_huggingface(
    model_info,
    private=True,
):

    print("=" * 80)
    print(f"Uploading {model_info['model_type']} Model")
    print("=" * 80)

    repo_id = f"{HF_USER_NAME}/{model_info['repo_name']}"

    create_repo(
        repo_id=repo_id,
        private=private,
        exist_ok=True,
    )

    model_info["model"].push_to_hub(repo_id)
    model_info["tokenizer"].push_to_hub(repo_id)

    print(f"\n✓ Uploaded successfully to:")
    print(f"https://huggingface.co/{repo_id}")

    print("=" * 80)

    return repo_id

### Common Hugging Face Merged Model Upload Function

In [38]:
def upload_merged_model_to_huggingface(
    model_info,
    private=True,
):

    print("=" * 80)
    print(f"Uploading Merged {model_info['model_type']} Model")
    print("=" * 80)

    # Merge LoRA adapter into the base model
    merged_model = model_info["model"].merge_and_unload()

    merged_repo_name = f"{model_info['repo_name']}-Merged"
    repo_id = f"{HF_USER_NAME}/{merged_repo_name}"

    create_repo(
        repo_id=repo_id,
        private=private,
        exist_ok=True,
    )

    merged_model.push_to_hub(repo_id)
    model_info["tokenizer"].push_to_hub(repo_id)

    print(f"\n✓ Uploaded successfully to:")
    print(f"https://huggingface.co/{repo_id}")

    print("=" * 80)

    return repo_id

### Upload LoRA to HF

In [20]:
upload_to_huggingface(lora) # Upload Adapter Model

Uploading LoRA Model


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Akay2026/University-AI-LoRA


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.



✓ Uploaded successfully to:
https://huggingface.co/Akay2026/University-AI-LoRA


'Akay2026/University-AI-LoRA'

In [21]:
upload_merged_model_to_huggingface(lora) # Upload Merged Model

Uploading Merged LoRA Model


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.



✓ Uploaded successfully to:
https://huggingface.co/Akay2026/University-AI-LoRA-Merged


'Akay2026/University-AI-LoRA-Merged'

## QLoRA Training

In [22]:
# ==============================================================================
# Train QLoRA Model
# ==============================================================================
#
# This section fine-tunes the base model using QLoRA.
#
# QLoRA loads the base model in 4-bit precision, significantly reducing GPU
# memory usage while achieving performance comparable to standard LoRA.
#
# The trained adapter is saved locally for later evaluation and upload.
#
# ==============================================================================

print("=" * 80)
print("Training QLoRA Model")
print("=" * 80)

# ------------------------------------------------------------------------------
# Train QLoRA Model
# ------------------------------------------------------------------------------

qlora = train_sft(

    use_4bit=True,

    local_model_name="University-AI-QLoRA",

)

print()

print("✓ QLoRA training completed successfully.")

print()

print(f"Model saved at:")

print(qlora["path"])

print("=" * 80)

Training QLoRA Model
Training QLoRA Model
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,3.704221
20,2.778035
30,1.637127
40,0.600798
50,0.173333
60,0.093630
70,0.075281
80,0.065927
90,0.062480
100,0.060509


Unsloth: Restored added_tokens_decoder metadata in models/University-AI-QLoRA/tokenizer_config.json.



✓ Adapter saved to
models/University-AI-QLoRA


✓ QLoRA training completed successfully.

Model saved at:
models/University-AI-QLoRA


### Test QLora

In [23]:
question = "How can I apply for a merit scholarship?"

response = generate_response(
    model_path=qlora["path"],
    questions=question
)
print(question)
print(response)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-QLoRA as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How can I apply for a merit scholarship?
system

Cutting Knowledge Date: December 2023
Today Date: 14 Jul 2026

You are Global Tech University's AI Student Assistant.user

How can I apply for a merit scholarship?assistant

Applying for a merit scholarship requires a thorough understanding of the institutional policies and formal verification pathways. At our university, the process is structured to protect academic integrity and ensure that all student records are updated chronically. Here is a step-by-step guide on how to execute this process smoothly:

1. Review the official university website to confirm the specific deadlines and requirements for merit scholarships.
2. Authenticate your identity on the portal using your primary university login and navigate directly to the formal application section.
3. Review the available scholarship types, eligibility criteria, and mandatory document requirements to ensure your file meets full compliance standards.
4. Complete the application for

### Upload QLoRA Model to Hugging Face

In [24]:
# Upload QLoRA Adapter Model to Hugging Face
upload_to_huggingface(qlora)

Uploading QLoRA Model


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Akay2026/University-AI-QLoRA


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.



✓ Uploaded successfully to:
https://huggingface.co/Akay2026/University-AI-QLoRA


'Akay2026/University-AI-QLoRA'

In [39]:
# Upload merged model to HF
upload_merged_model_to_huggingface(qlora)

Uploading Merged QLoRA Model


NotImplementedError: 

## Direct Preference Optimization (DPO)

In [40]:
# Verification of Version

import torch
import transformers
import trl
import peft
import datasets
import unsloth

print("=" * 60)
print("Environment Verification")
print("=" * 60)
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"PEFT         : {peft.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

Environment Verification
PyTorch      : 2.10.0+cu128
Transformers : 5.5.0
TRL          : 0.24.0
PEFT         : 0.19.1
Datasets     : 4.3.0
CUDA         : True
GPU          : Tesla T4


### Load the SFT Model (LoRA)

In [41]:
# ============================================================
# Load the SFT LoRA Model
# ============================================================

# Path where the SFT LoRA adapter was saved in Module 4
# SFT_MODEL_PATH = "models/lora_sft"
SFT_MODEL_PATH = "models/University-AI-LoRA"
print("=" * 60)
print("Loading SFT LoRA Model...")
print("=" * 60)

# Load base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_PATH,
    max_seq_length=2048,
    load_in_4bit=False,      # LoRA model (not QLoRA)
)

print("\n✅ SFT LoRA model loaded successfully.")

print(f"\nModel Path : {SFT_MODEL_PATH}")
print(f"Model Type : LoRA")
print(f"Max Length : 2048")

Loading SFT LoRA Model...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-LoRA as a legacy tokenizer.



✅ SFT LoRA model loaded successfully.

Model Path : models/University-AI-LoRA
Model Type : LoRA
Max Length : 2048


### Load and Validate the DPO Dataset

In [42]:
# ============================================================
# Load DPO Dataset
# ============================================================

from datasets import load_from_disk

print("=" * 60)
print("Loading DPO Dataset...")
print("=" * 60)

# Load the processed Hugging Face dataset
dpo_dataset = load_from_disk(DPO_DATASET_DIR)

print(f"Total Records : {len(dpo_dataset):,}")

print("\nDataset Features:")
print(dpo_dataset.features)

print("\nSample Record")
print("-" * 60)

print("Prompt:")
print(dpo_dataset[0]["prompt"])

print("\nChosen:")
print(dpo_dataset[0]["chosen"])

print("\nRejected:")
print(dpo_dataset[0]["rejected"])

print("\n✅ DPO Dataset loaded successfully.")

Loading DPO Dataset...
Total Records : 1,000

Dataset Features:
{'prompt': Value('string'), 'chosen': Value('string'), 'rejected': Value('string')}

Sample Record
------------------------------------------------------------
Prompt:
What is the official institutional framework, policy workflow, and compliance standard for managing admissions ensuring that all procedural risks are minimized and that the student maintains full structural alignment with university regulations. [Case Identifier Ref: #10001]

Chosen:
**Brief Overview:**
Navigating the administrative matrix for Admissions requires a thorough understanding of localized department workflows and centralized registry rules. The institution maintains an active verification process to ensure that all student records, transactions, and milestones are logged with full audit trails, minimizing procedural delays and protecting student progression.

**Detailed Advisor Explanation:**
Applying structural regulation mapping code ADM-0001 d

### Create the Reference Model

In [43]:
# ============================================================
# Module 7 - Cell 4
# Create Reference Model
# ============================================================

import copy

print("=" * 60)
print("Creating Reference Model...")
print("=" * 60)

# Create a frozen copy of the SFT model
ref_model = copy.deepcopy(model)

# Freeze all parameters
for param in ref_model.parameters():
    param.requires_grad = False

# Put reference model in evaluation mode
ref_model.eval()

print("✅ Reference model created successfully.")
print("Reference model parameters are frozen.")

Creating Reference Model...
✅ Reference model created successfully.
Reference model parameters are frozen.


In [44]:
# ============================================================
# Configure DPO Training
# ============================================================
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(

    # Output
    output_dir="./outputs_dpo",

    # Training
    num_train_epochs=1,
    learning_rate=5e-6,

    # Batch size
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Sequence lengths
    max_prompt_length=512,
    max_length=1024,

    # DPO
    beta=0.1,

    # Logging
    logging_steps=10,
    save_strategy="epoch",

    # Performance
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),

    report_to="none",
)

print("=" * 60)
print("DPO Configuration")
print("=" * 60)

print(dpo_config)

DPO Configuration
UnslothDPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
base_model_attribute_name=model,
batch_eval_metrics=False,
beta=0.1,
bf16=False,
bf16_full_eval=False,
data_seed=3407,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_num_proc=8,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_dropout=True,
disable_tqdm=False,
discopop_tau=0.05,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=2,
eval_delay=0,
eval_do_conca

### Create the DPO Trainer

In [45]:
# ============================================================
# Module 7 - Cell 6
# Create DPO Trainer
# ============================================================

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("=" * 60)
print("DPO Trainer created successfully.")
print("=" * 60)

print(f"Training Samples : {len(dpo_dataset):,}")
print(f"Tokenizer        : {tokenizer.__class__.__name__}")

Extracting prompt in train dataset (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

DPO Trainer created successfully.
Training Samples : 1,000
Tokenizer        : TokenizersBackend


### Start DPO Training

In [46]:
# ============================================================
# Module 7 - Cell 7
# Train the DPO Model
# ============================================================

print("=" * 60)
print("Starting DPO Training...")
print("=" * 60)

train_result = trainer.train()
dpo_model = model # Just adding refernce to trained model for easy reference

print("\n" + "=" * 60)
print("DPO Training Completed Successfully!")
print("=" * 60)

print("\nTraining Metrics:")
print(train_result.metrics)

Starting DPO Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
10,0.561490,0.004605,-0.472491,0.300000,0.477096,-79.328529,-1827.605225,-1.428782,-2.137747
20,0.001204,0.132024,-17.903181,1.000000,18.035206,-79.696915,-1991.699463,-1.219172,-2.123928
30,0.000000,0.153157,-42.599712,1.000000,42.752872,-71.194313,-2203.916260,-0.978251,-2.006858
40,0.000000,0.046717,-54.013416,1.000000,54.060131,-76.704147,-2365.084717,-0.838470,-1.917616
50,0.000000,-0.109427,-56.716469,1.000000,56.607044,-77.590691,-2395.544434,-0.796553,-1.854321
60,0.000000,-0.161444,-56.972736,1.000000,56.811291,-72.683487,-2388.527344,-0.818611,-1.823712
70,0.000000,-0.087204,-58.337639,1.000000,58.250446,-76.665184,-2373.433105,-0.788101,-1.850354
80,0.000000,0.047693,-60.126087,1.000000,60.173779,-76.861000,-2384.310059,-0.769795,-1.897657
90,0.000000,-0.138050,-58.573875,1.000000,58.435841,-81.176880,-2420.153809,-0.777654,-1.830381
100,0.000000,-0.151797,-57.918560,1.000000,57.766754,-78.550217,-2404.574951,-0.789293,-1.813499


Unsloth: Restored added_tokens_decoder metadata in ./outputs_dpo/checkpoint-125/tokenizer_config.json.



DPO Training Completed Successfully!

Training Metrics:
{'train_runtime': 763.7311, 'train_samples_per_second': 1.309, 'train_steps_per_second': 0.164, 'total_flos': 0.0, 'train_loss': 0.045015486255288166, 'epoch': 1.0}


### Save the DPO LoRA Adapter

In [47]:
# ============================================================
# Save the DPO LoRA Model
# ============================================================

DPO_MODEL_DIR = "models/University-AI-DPO"

print("=" * 60)
print("Saving DPO LoRA Model...")
print("=" * 60)

# Save LoRA adapter
dpo_model.save_pretrained(DPO_MODEL_DIR)

# Save tokenizer
tokenizer.save_pretrained(DPO_MODEL_DIR)

print(f"✅ DPO LoRA model saved to: {DPO_MODEL_DIR}")

Saving DPO LoRA Model...


Unsloth: Restored added_tokens_decoder metadata in models/University-AI-DPO/tokenizer_config.json.


✅ DPO LoRA model saved to: models/University-AI-DPO


### Create DPO Model Info

In [51]:
dpo_model_info = {
    "trainer": trainer,
    "model": trainer.model,
    "tokenizer": tokenizer,
    "path": "models/University-AI-DPO",

    # Metadata
    "model_name": "University-AI-DPO",
    "base_model": "University-AI-LoRA",
    "model_type": "DPO LoRA",
    "repo_name": "University-AI-DPO",
}

### Test the DPO Model

In [52]:
question = "How can I apply for a merit scholarship?"

response = generate_response(
    model_path=dpo_model_info["path"],
    questions=question
)
print(question)
print(response)

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-DPO as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


How can I apply for a merit scholarship?
system

Cutting Knowledge Date: December 2023
Today Date: 14 Jul 2026

You are Global Tech University's AI Student Assistant.user

How can I apply for a merit scholarship?assistant

Applying for a merit scholarship at our university involves an administrative workflow designed to streamline standard institutional procedures while maintaining the rigorous criteria expected of the academic community. For merit scholarships, the process typically begins with a formal request submission form completed entirely by the student on the university website. These forms are evaluated by specialized administrative teams that assess the validity of academic timelines, balance of credits, and compliance with institutional bylaws. Once the formal submission phase completes, the approved request is typically scheduled a final administrative hearing or a desk audit. During this final audit, the student's file is reviewed from a administrative perspective, highli

### Push DPO LoRA Model to Hugging Face

In [49]:
upload_to_huggingface(dpo_model_info)

Uploading DPO LoRA Model


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Akay2026/University-AI-DPO


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.



✓ Uploaded successfully to:
https://huggingface.co/Akay2026/University-AI-DPO


'Akay2026/University-AI-DPO'

In [50]:
upload_merged_model_to_huggingface(dpo_model_info)

Uploading Merged DPO LoRA Model


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.



✓ Uploaded successfully to:
https://huggingface.co/Akay2026/University-AI-DPO-Merged


'Akay2026/University-AI-DPO-Merged'

### Evaluate the DPO Model on Test Prompts

In [53]:
# ============================================================
# Evaluate Model on Test Dataset
# ============================================================

NUM_QUESTIONS = 2

# questions = [
#     row["question"]
#     for row in test_dataset[:NUM_QUESTIONS]
# ]

questions = test_dataset["question"][:NUM_QUESTIONS]

responses = generate_response(
    model_path=dpo_model_info["path"],
    questions=questions,
    num_questions_to_run=NUM_QUESTIONS,
)

print("=" * 100)
print("Model Evaluation")
print("=" * 100)

for i, response in enumerate(responses):

    print("=" * 100)
    print(f"Test Case {i + 1}")
    print("=" * 100)

    print(f"Category : {test_dataset[i]['category']}")

    print("\nQuestion")
    print("-" * 100)
    print(test_dataset[i]["question"])

    print("\nExpected Answer")
    print("-" * 100)
    print(test_dataset[i]["expected_answer"])

    print("\nModel Response")
    print("-" * 100)
    print(response)

    print("\n")

==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-DPO as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Model Evaluation
Test Case 1
Category : Admissions

Question
----------------------------------------------------------------------------------------------------
Could someone explain what happens if a student misses the window for conditional acceptance offer reviews due to unforeseen conflicts?

Expected Answer
----------------------------------------------------------------------------------------------------
**Official Advisor Guidance:** When addressing conditional acceptance offer reviews inside the admissions framework, the university manual specifies that every record must maintain structural validation. The university approaches these scenarios through an objective administrative lens designed to enforce institutional bylaws while accommodating legitimate, third-party verified disruptions. When a student encounters a barrier in this operational branch, the registrar opens an inquiry file to audit the account history against system transaction logs. If a variance is justified u

### DPO Training Summary

In [56]:
# ============================================================
# DPO Training Summary
# ============================================================

print(f"{'Model Name':25}: {dpo_model_info['model_name']}")
print(f"{'Model Type':25}: {dpo_model_info['model_type']}")
print(f"{'Base Model':25}: {dpo_model_info['base_model']}")
print(f"{'Saved Model Path':25}: {dpo_model_info['path']}")
print(f"{'Hugging Face Repo':25}: {HF_USER_NAME}/{dpo_model_info['repo_name']}")

print("\nTraining Metrics")
print("-" * 80)

metrics = trainer.state.log_history[-1]

print(f"{'Training Loss':25}: {metrics.get('train_loss', 'N/A')}")
print(f"{'Epochs':25}: {metrics.get('epoch', 'N/A')}")
print(f"{'Training Runtime (sec)':25}: {metrics.get('train_runtime', 'N/A')}")
print(f"{'Samples / Second':25}: {metrics.get('train_samples_per_second', 'N/A')}")
print(f"{'Steps / Second':25}: {metrics.get('train_steps_per_second', 'N/A')}")

print("=" * 80)


Model Name               : University-AI-DPO
Model Type               : DPO LoRA
Base Model               : University-AI-LoRA
Saved Model Path         : models/University-AI-DPO
Hugging Face Repo        : Akay2026/University-AI-DPO

Training Metrics
--------------------------------------------------------------------------------
Training Loss            : 0.045015486255288166
Epochs                   : 1.0
Training Runtime (sec)   : 763.7311
Samples / Second         : 1.309
Steps / Second           : 0.164


## Model Benchmarking and Comparison

### Load Evaluation Benchmark Dataset

In [57]:
# ============================================================
# Load Benchmark Evaluation Dataset
# ============================================================

from datasets import Dataset
import pandas as pd

EVALUATION_FILE = DATA_DIR / "University_EVAL_Dataset.xlsx"

evaluation_df = pd.read_excel(EVALUATION_FILE)

evaluation_dataset = Dataset.from_pandas(
    evaluation_df,
    preserve_index=False,
)

print("=" * 80)
print("Benchmark Evaluation Dataset Loaded")
print("=" * 80)

print(f"Total Evaluation Prompts : {len(evaluation_dataset):,}")

print("\nColumns")
print("-" * 80)
print(evaluation_dataset.column_names)

print("\nSample Records")
print("-" * 80)

evaluation_df.head()

Benchmark Evaluation Dataset Loaded
Total Evaluation Prompts : 500

Columns
--------------------------------------------------------------------------------
['id', 'difficulty', 'question']

Sample Records
--------------------------------------------------------------------------------


,id,difficulty,question
0,EVAL-0001,Easy,Who is the primary point of contact or office ...
1,EVAL-0002,Medium,If my initial application for a policy adjustm...
2,EVAL-0003,Hard,Can you outline the comprehensive strategy for...
3,EVAL-0004,Easy,Where on the student portal can I locate the s...
4,EVAL-0005,Medium,My academic dashboard is showing an unexpected...


### Explore Benchmark Dataset

In [61]:
# ============================================================
# Explore Benchmark Dataset
# ============================================================

print("=" * 80)
print("Benchmark Dataset Statistics")
print("=" * 80)

print(f"Total Evaluation Prompts : {len(evaluation_df):,}")

print("\nDifficulty Distribution")
print("-" * 80)
print(evaluation_df["difficulty"].value_counts())

print("\nSample Benchmark Prompts")
print("-" * 80)

for difficulty in sorted(evaluation_df["difficulty"].unique()):

    print(f"\n{difficulty} Questions")
    print("-" * 40)

    sample_questions = evaluation_df[
        evaluation_df["difficulty"] == difficulty
    ].head(3)

    for _, row in sample_questions.iterrows():
        print(f"• {row['question']}")

Benchmark Dataset Statistics
Total Evaluation Prompts : 500

Difficulty Distribution
--------------------------------------------------------------------------------
difficulty
Easy      167
Medium    167
Hard      166
Name: count, dtype: int64

Sample Benchmark Prompts
--------------------------------------------------------------------------------

Easy Questions
----------------------------------------
• Who is the primary point of contact or office supervisor responsible for approving basic requests in the admissions department? [Evaluation Ticket: #70001]
• Where on the student portal can I locate the standard forms for scholarships and what is the general filing deadline? [Evaluation Ticket: #70004]
• What are the standard operational hours for the front desk window that handles physical submissions for payment plans? [Evaluation Ticket: #70007]

Hard Questions
----------------------------------------
• Can you outline the comprehensive strategy for structural credit reconciliati

### Model Info of Models to Compare

In [62]:
foundation_model_info = { # Hugging Face pretrained model
    "path": MODEL_NAME,
    "model_name": MODEL_NAME,
    "model_type": "Foundation Model",
}
lora_model_info = lora # SFT model
qlora_model_info = qlora # QLoRA model
# dpo_model_info # DPO model trained on LorA SFT Model

model_info_list = [foundation_model_info, lora_model_info, qlora_model_info, dpo_model_info]

target_keys = ["path", "model_name", "model_type"]

for info in model_info_list:
    # Use .get() to prevent KeyError if a dictionary is missing a specific key
    for key in target_keys:
        clean_label = key.replace('_', ' ').capitalize()
        value = info.get(key, "Not specified") 
        print(f"{clean_label}: {value}")
    print("-" * 20)

Path: unsloth/Llama-3.2-1B-Instruct
Model name: unsloth/Llama-3.2-1B-Instruct
Model type: Foundation Model
--------------------
Path: models/University-AI-LoRA
Model name: University-AI-LoRA
Model type: LoRA
--------------------
Path: models/University-AI-QLoRA
Model name: University-AI-QLoRA
Model type: QLoRA
--------------------
Path: models/University-AI-DPO
Model name: University-AI-DPO
Model type: DPO LoRA
--------------------


### Generate Responses from All Models

In [60]:
# ============================================================
# Generate Responses from All Models
# ============================================================

NUM_QUESTIONS = 3

# --------------------------------------------------------------------------
# Foundation Model Information
# --------------------------------------------------------------------------

# foundation_model_info = {
#     "path": MODEL_NAME,
#     "model_name": MODEL_NAME,
#     "model_type": "Foundation Model",
# }

# --------------------------------------------------------------------------
# Prepare Benchmark Questions
# --------------------------------------------------------------------------

questions = evaluation_df["question"].tolist()

print("=" * 80)
print("Generating Benchmark Responses")
print("=" * 80)

# --------------------------------------------------------------------------
# Foundation Model
# --------------------------------------------------------------------------

print("\nGenerating responses from Foundation Model...")

foundation_responses = generate_response(
    model_path=foundation_model_info["path"],
    questions=questions,
    num_questions_to_run=NUM_QUESTIONS,
)

print("✓ Foundation Model completed.")

# --------------------------------------------------------------------------
# LoRA Model
# --------------------------------------------------------------------------

print("\nGenerating responses from LoRA Model...")

lora_responses = generate_response(
    model_path=lora_model_info["path"],
    questions=questions,
    num_questions_to_run=NUM_QUESTIONS,
)

print("✓ LoRA Model completed.")

# --------------------------------------------------------------------------
# QLoRA Model
# --------------------------------------------------------------------------

print("\nGenerating responses from QLoRA Model...")

qlora_responses = generate_response(
    model_path=qlora_model_info["path"],
    questions=questions,
    num_questions_to_run=NUM_QUESTIONS,
)

print("✓ QLoRA Model completed.")

# --------------------------------------------------------------------------
# DPO Model
# --------------------------------------------------------------------------

print("\nGenerating responses from DPO Model...")

dpo_responses = generate_response(
    model_path=dpo_model_info["path"],
    questions=questions,
    num_questions_to_run=NUM_QUESTIONS,
)

print("✓ DPO Model completed.")

print("\n" + "=" * 80)
print("Benchmark Response Generation Completed Successfully!")
print("=" * 80)

Generating Benchmark Responses

Generating responses from Foundation Model...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-Instruct as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Foundation Model completed.

Generating responses from LoRA Model...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

Unsloth: Will load models/University-AI-QLoRA as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ QLoRA Model completed.

Generating responses from DPO Model...
==((====))==  Unsloth 2026.7.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load models/University-AI-DPO as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ DPO Model completed.

Benchmark Response Generation Completed Successfully!


### Compare Responses from All Models

In [63]:
# ============================================================
# Compare Model Responses
# ============================================================

for i in range(NUM_QUESTIONS):

    print("=" * 100)
    print(f"Benchmark Question {i+1}")
    print("=" * 100)

    print(f"Difficulty : {evaluation_df.iloc[i]['difficulty']}")

    print("\nQuestion")
    print("-" * 100)
    print(evaluation_df.iloc[i]["question"])

    print("\nFoundation Model")
    print("-" * 100)
    print(foundation_responses[i])

    print("\nLoRA Model")
    print("-" * 100)
    print(lora_responses[i])

    print("\nQLoRA Model")
    print("-" * 100)
    print(qlora_responses[i])

    print("\nDPO Model")
    print("-" * 100)
    print(dpo_responses[i])

    print("\n")

Benchmark Question 1
Difficulty : Easy

Question
----------------------------------------------------------------------------------------------------
Who is the primary point of contact or office supervisor responsible for approving basic requests in the admissions department? [Evaluation Ticket: #70001]

Foundation Model
----------------------------------------------------------------------------------------------------
system

Cutting Knowledge Date: December 2023
Today Date: 14 Jul 2026

You are Global Tech University's AI Student Assistant.user

Who is the primary point of contact or office supervisor responsible for approving basic requests in the admissions department? [Evaluation Ticket: #70001]assistant

I cannot verify who the primary point of contact or office supervisor is for the admissions department at Global Tech University.

LoRA Model
----------------------------------------------------------------------------------------------------
system

Cutting Knowledge Date: Dec

### Create Benchmark Comparison DataFrame

In [64]:
# ============================================================
# Create Benchmark Comparison DataFrame
# ============================================================

comparison_df = evaluation_df.head(NUM_QUESTIONS).copy()

comparison_df["Foundation Response"] = foundation_responses
comparison_df["LoRA Response"] = lora_responses
comparison_df["QLoRA Response"] = qlora_responses
comparison_df["DPO Response"] = dpo_responses

print("=" * 80)
print("Benchmark Comparison Results")
print("=" * 80)

print(f"Total Questions Compared : {len(comparison_df)}")

comparison_df

Benchmark Comparison Results
Total Questions Compared : 3


,id,difficulty,question,Foundation Response,LoRA Response,QLoRA Response,DPO Response
0,EVAL-0001,Easy,Who is the primary point of contact or office ...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
1,EVAL-0002,Medium,If my initial application for a policy adjustm...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...
2,EVAL-0003,Hard,Can you outline the comprehensive strategy for...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...


### Save Benchmark Comparison Report to Excel (Benchmark_Comparison.xls)

In [65]:
# ============================================================
# Save Benchmark Comparison Report
# ============================================================

comparison_df["Foundation Length"] = (
    comparison_df["Foundation Response"]
    .str.split()
    .str.len()
)

comparison_df["LoRA Length"] = (
    comparison_df["LoRA Response"]
    .str.split()
    .str.len()
)

comparison_df["QLoRA Length"] = (
    comparison_df["QLoRA Response"]
    .str.split()
    .str.len()
)

comparison_df["DPO Length"] = (
    comparison_df["DPO Response"]
    .str.split()
    .str.len()
)

comparison_file = OUTPUT_DIR / "Benchmark_Comparison.xlsx"

comparison_df.to_excel(
    comparison_file,
    index=False,
)

print("=" * 80)
print("Benchmark Comparison Report Saved Successfully")
print("=" * 80)

print(f"\nLocation:\n{comparison_file}")

comparison_df.head()

Benchmark Comparison Report Saved Successfully

Location:
output/Benchmark_Comparison.xlsx


,id,difficulty,question,Foundation Response,LoRA Response,QLoRA Response,DPO Response,Foundation Length,LoRA Length,QLoRA Length,DPO Length
0,EVAL-0001,Easy,Who is the primary point of contact or office ...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,62,86,194,68
1,EVAL-0002,Medium,If my initial application for a policy adjustm...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,197,211,126,106
2,EVAL-0003,Hard,Can you outline the comprehensive strategy for...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,system\n\nCutting Knowledge Date: December 202...,194,209,211,209


### Benchmark Summary

In [67]:
# ============================================================
# Benchmark Summary
# ============================================================

print("=" * 80)
print("Benchmark Summary")
print("=" * 80)

print(f"Evaluation Questions : {len(comparison_df)}")

print("\nAverage Response Length (Words)")
print("-" * 80)

avg_foundation = comparison_df["Foundation Length"].mean()
avg_lora = comparison_df["LoRA Length"].mean()
avg_qlora = comparison_df["QLoRA Length"].mean()
avg_dpo = comparison_df["DPO Length"].mean()

print(f"Foundation Model : {avg_foundation:.1f}")
print(f"LoRA Model       : {avg_lora:.1f}")
print(f"QLoRA Model      : {avg_qlora:.1f}")
print(f"DPO Model        : {avg_dpo:.1f}")

print("\nLongest Response")
print("-" * 80)

print(f"Foundation : {comparison_df['Foundation Length'].max()} words")
print(f"LoRA       : {comparison_df['LoRA Length'].max()} words")
print(f"QLoRA      : {comparison_df['QLoRA Length'].max()} words")
print(f"DPO        : {comparison_df['DPO Length'].max()} words")

print("\nShortest Response")
print("-" * 80)

print(f"Foundation : {comparison_df['Foundation Length'].min()} words")
print(f"LoRA       : {comparison_df['LoRA Length'].min()} words")
print(f"QLoRA      : {comparison_df['QLoRA Length'].min()} words")
print(f"DPO        : {comparison_df['DPO Length'].min()} words")

print("\nGenerated Files")
print("-" * 80)

print(comparison_file)

print("\n" + "=" * 80)
print("Module 8 Completed Successfully")
print("=" * 80)

Benchmark Summary
Evaluation Questions : 3

Average Response Length (Words)
--------------------------------------------------------------------------------
Foundation Model : 151.0
LoRA Model       : 168.7
QLoRA Model      : 177.0
DPO Model        : 127.7

Longest Response
--------------------------------------------------------------------------------
Foundation : 197 words
LoRA       : 211 words
QLoRA      : 211 words
DPO        : 209 words

Shortest Response
--------------------------------------------------------------------------------
Foundation : 62 words
LoRA       : 86 words
QLoRA      : 126 words
DPO        : 68 words

Generated Files
--------------------------------------------------------------------------------
output/Benchmark_Comparison.xlsx

Module 8 Completed Successfully
